# 中芯国际 (688981.SH) 数据分析与可视化

本 Notebook 演示完整的金融数据获取与分析流程：
1. 通过 **Tushare Pro API** 获取中芯国际近一年日线行情
2. 数据清洗与预处理
3. 使用 **Plotly** 绘制交互式 K 线图（蜡烛图）
4. 绘制成交量柱状图
5. 统计分析

> 日期范围：约 2025-06 至 2026-07，共约 250 个交易日

## 0. 环境准备

安装所需依赖（如果尚未安装）：

In [ ]:
# !pip install tushare pandas plotly -q

## 1. 导入库并配置

In [ ]:
import tushare as ts
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

# 设置 Tushare Token（请替换为你自己的 Token）
TUSHARE_TOKEN = "YOUR_TUSHARE_TOKEN"
ts.set_token(TUSHARE_TOKEN)
pro = ts.pro_api()

# 股票信息
TS_CODE = "688981.SH"  # 中芯国际
STOCK_NAME = "中芯国际"

# 计算时间范围：近一年
end_date = datetime.now().strftime("%Y%m%d")
start_date = (datetime.now() - timedelta(days=380)).strftime("%Y%m%d")

print(f"数据获取范围: {start_date} ~ {end_date}")
print(f"股票代码: {TS_CODE}")

## 2. 获取日线行情数据

通过 Tushare `pro.daily()` 接口获取日级别 OHLCV 数据。

**字段说明**：
- `ts_code` — 股票代码
- `trade_date` — 交易日期
- `open` / `high` / `low` / `close` — 开/高/低/收
- `pre_close` — 前收盘价
- `change` — 涨跌额
- `pct_chg` — 涨跌幅（%）
- `vol` — 成交量（手）
- `amount` — 成交额（千元）

In [ ]:
# 调用 Tushare API
df = pro.daily(
    ts_code=TS_CODE,
    start_date=start_date,
    end_date=end_date
)

print(f"获取到 {len(df)} 条记录")
df.head()

### 2.1 如果 Tushare SDK 不可用，使用纯 HTTP API 方式

以下为备选方案，仅依赖 Python 标准库：

In [ ]:
# ===== 备选方案：使用 urllib 直接调用 Tushare HTTP API =====
# 当 tushare SDK 不可用时取消注释以下代码

# import urllib.request
# import json

# payload = json.dumps({
#     "api_name": "daily",
#     "token": TUSHARE_TOKEN,
#     "params": {"ts_code": TS_CODE, "start_date": start_date, "end_date": end_date},
#     "fields": "ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount"
# }).encode("utf-8")

# req = urllib.request.Request("https://api.tushare.pro", data=payload, 
#                              headers={"Content-Type": "application/json"})
# with urllib.request.urlopen(req, timeout=30) as resp:
#     result = json.loads(resp.read().decode("utf-8"))

# data = result["data"]
# df = pd.DataFrame(data["items"], columns=data["fields"])
# print(f"获取到 {len(df)} 条记录")
# df.head()

## 3. 数据清洗与预处理

In [ ]:
# 转换数据类型
numeric_cols = ['open', 'high', 'low', 'close', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 日期升序排列（Tushare 默认返回降序）
df = df.sort_values('trade_date').reset_index(drop=True)

# 格式化日期列
df['date'] = pd.to_datetime(df['trade_date'], format='%Y%m%d')

# 计算成交额（万元）和成交量（手→万股）便于阅读
df['amount_wan'] = df['amount'] / 10    # 千元 → 万元
df['vol_wan'] = df['vol'] / 100          # 手 → 万股（100手=1万股）

print(f"清洗后数据: {len(df)} 条")
print(f"日期范围: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"价格范围: ¥{df['low'].min():.2f} ~ ¥{df['high'].max():.2f}")
df[['trade_date', 'open', 'close', 'high', 'low', 'pct_chg', 'vol']].head()

### 3.1 缺失值检查

In [ ]:
# 检查是否有缺失值
missing = df.isnull().sum()
print("各列缺失值数量:")
print(missing[missing > 0] if missing.sum() > 0 else "无缺失值 ✓")

## 4. 描述性统计

先看看数据的整体情况。

In [ ]:
# 关键指标统计
stats = df[['open', 'high', 'low', 'close', 'pct_chg', 'vol_wan', 'amount_wan']].describe()
stats.round(2)

In [ ]:
# 近一年整体表现
first_close = df['pre_close'].iloc[0]   # 期初前收盘
last_close = df['close'].iloc[-1]       # 最新收盘
total_return = (last_close - first_close) / first_close * 100
volatility = df['pct_chg'].std()        # 日波动率

print(f"{'='*40}")
print(f"  {STOCK_NAME} ({TS_CODE}) 近一年表现")
print(f"{'='*40}")
print(f"  期初前收盘价:  ¥{first_close:.2f}")
print(f"  最新收盘价:    ¥{last_close:.2f}")
print(f"  区间涨跌幅:    {total_return:+.2f}%")
print(f"  区间最高价:    ¥{df['high'].max():.2f}")
print(f"  区间最低价:    ¥{df['low'].min():.2f}")
print(f"  日均波动率:    {volatility:.2f}%")
print(f"  日均成交量:    {df['vol_wan'].mean():.0f} 万股")
print(f"  总交易日数:    {len(df)}")
print(f"  上涨天数:      {(df['pct_chg'] > 0).sum()} ({((df['pct_chg'] > 0).sum()/len(df)*100):.1f}%)")
print(f"  下跌天数:      {(df['pct_chg'] < 0).sum()} ({((df['pct_chg'] < 0).sum()/len(df)*100):.1f}%)")

## 5. 交互式 K 线图 + 成交量图

使用 **Plotly** 绘制专业的金融可视化面板：
- 上方：K 线蜡烛图（Candlestick）
- 下方：成交量柱状图
- 红涨绿跌（中国股市配色）
- 支持缩放、悬浮提示、区间选择

In [ ]:
# 准备数据
dates = df['date']

# 成交量颜色：收盘 >= 开盘 为红色，否则绿色
vol_colors = ['#e53935' if c >= o else '#43a047' 
              for c, o in zip(df['close'], df['open'])]

# 创建子图（2行1列），K线占70%，成交量占30%
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.7, 0.3],
    subplot_titles=(f"{STOCK_NAME} ({TS_CODE}) — K线图", "成交量")
)

# ---- K 线图 ----
fig.add_trace(
    go.Candlestick(
        x=dates,
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        name='K线',
        increasing=dict(line=dict(color='#e53935'), fillcolor='#e53935'),
        decreasing=dict(line=dict(color='#43a047'), fillcolor='#43a047'),
        hovertext=[f"日期: {d.strftime('%Y-%m-%d')}<br>"
                   f"开: ¥{o:.2f}  收: ¥{c:.2f}<br>"
                   f"高: ¥{h:.2f}  低: ¥{l:.2f}<br>"
                   f"涨跌幅: {p:+.2f}%"
                   for d, o, c, h, l, p in zip(
                       dates, df['open'], df['close'],
                       df['high'], df['low'], df['pct_chg'])],
        hoverinfo='text'
    ),
    row=1, col=1
)

# 添加移动均线 MA10
df['ma10'] = df['close'].rolling(window=10).mean()
fig.add_trace(
    go.Scatter(
        x=dates, y=df['ma10'],
        mode='lines',
        line=dict(color='#ff9800', width=1.5, dash='solid'),
        name='MA10',
        hovertemplate='MA10: ¥%{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)

# 添加移动均线 MA30
df['ma30'] = df['close'].rolling(window=30).mean()
fig.add_trace(
    go.Scatter(
        x=dates, y=df['ma30'],
        mode='lines',
        line=dict(color='#2196f3', width=1.5, dash='dash'),
        name='MA30',
        hovertemplate='MA30: ¥%{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)

# ---- 成交量图 ----
fig.add_trace(
    go.Bar(
        x=dates,
        y=df['vol_wan'],
        marker_color=vol_colors,
        name='成交量(万股)',
        hovertemplate='日期: %{x|%Y-%m-%d}<br>成交量: %{y:,.0f} 万股<extra></extra>'
    ),
    row=2, col=1
)

# ---- 布局配置 ----
fig.update_layout(
    title=dict(
        text=f"<b>{STOCK_NAME}</b> ({TS_CODE}) 日线行情",
        font=dict(size=20),
        x=0.5
    ),
    xaxis_rangeslider_visible=False,  # 隐藏默认的底部滑块
    height=750,
    hovermode='x unified',
    paper_bgcolor='white',
    plot_bgcolor='#fafafa',
    margin=dict(l=60, r=30, t=70, b=40),
    showlegend=True,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    )
)

# 坐标轴美化
fig.update_xaxes(
    showgrid=True, gridcolor='#eee', 
    showspikes=True, spikecolor='#999', spikethickness=1,
    spikedash='dot', spikemode='across',
    rangeslider=dict(visible=False)
)
fig.update_yaxes(
    title_text='价格 (¥)',
    showgrid=True, gridcolor='#eee',
    row=1, col=1
)
fig.update_yaxes(
    title_text='成交量 (万股)',
    showgrid=True, gridcolor='#eee',
    row=2, col=1
)

fig.show()

## 6. 涨跌幅分布分析

In [ ]:
# 涨跌幅分布直方图
fig_hist = go.Figure()

fig_hist.add_trace(go.Histogram(
    x=df['pct_chg'],
    nbinsx=40,
    marker=dict(
        color='rgba(229, 57, 53, 0.7)',
        line=dict(color='#e53935', width=1)
    ),
    name='日涨跌幅分布'
))

# 添加均值线
mean_ret = df['pct_chg'].mean()
fig_hist.add_vline(x=mean_ret, line_dash='dash', line_color='#333',
                   annotation_text=f'均值 {mean_ret:+.2f}%',
                   annotation_position='top')
fig_hist.add_vline(x=0, line_dash='dot', line_color='#999',
                   annotation_text='0%')

fig_hist.update_layout(
    title=f'{STOCK_NAME} 日涨跌幅分布',
    xaxis_title='涨跌幅 (%)',
    yaxis_title='天数',
    height=400,
    bargap=0.05,
    paper_bgcolor='white',
    plot_bgcolor='#fafafa'
)

fig_hist.show()

## 7. 月度收益率热力图

In [ ]:
# 计算月度收益率
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# 按月计算累计收益率
monthly = df.groupby(['year', 'month']).agg(
    月涨跌幅=('pct_chg', 'sum'),
    月均成交量=('vol_wan', 'mean'),
    交易天数=('date', 'count')
).reset_index()

monthly['月份标签'] = monthly['year'].astype(str) + '-' + monthly['month'].astype(str).str.zfill(2)
monthly['月涨跌幅'] = monthly['月涨跌幅'].round(2)

display(monthly[['月份标签', '月涨跌幅', '月均成交量', '交易天数']])

In [ ]:
# 月度收益柱状图
fig_monthly = go.Figure()

colors = ['#e53935' if v > 0 else '#43a047' for v in monthly['月涨跌幅']]

fig_monthly.add_trace(go.Bar(
    x=monthly['月份标签'],
    y=monthly['月涨跌幅'],
    marker_color=colors,
    text=monthly['月涨跌幅'].apply(lambda x: f'{x:+.1f}%'),
    textposition='outside',
    textfont=dict(size=11),
    hovertemplate='%{x}<br>月涨跌幅: %{y:+.2f}%<br>日均成交量: %{customdata:,.0f} 万股<extra></extra>',
    customdata=monthly['月均成交量']
))

fig_monthly.update_layout(
    title=f'{STOCK_NAME} 月度涨跌幅',
    xaxis_title='月份',
    yaxis_title='涨跌幅 (%)',
    height=400,
    paper_bgcolor='white',
    plot_bgcolor='#fafafa',
    yaxis=dict(zeroline=True, zerolinecolor='#999', zerolinewidth=1)
)

fig_monthly.show()

## 8. 导出数据

将清洗后的数据保存为 CSV 文件。

In [ ]:
# 保存到 CSV
output_path = "smic_daily_data_cleaned.csv"
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"数据已导出: {output_path}")
print(f"文件大小: {len(df)} 行 × {len(df.columns)} 列")

## 9. 总结

本 Notebook 完成了以下工作：

| 步骤 | 内容 |
|------|------|
| 数据获取 | 通过 Tushare Pro API 获取近一年日线数据 |
| 数据清洗 | 类型转换、排序、格式化日期、计算辅助列 |
| K线图 | Plotly Candlestick + MA10/MA30 均线 |
| 成交量图 | 红涨绿跌配色柱状图 |
| 统计分析 | 描述性统计、涨跌幅分布、月度收益 |
| 数据导出 | 保存清洗后的 CSV 文件 |

> 💡 提示：替换 `TUSHARE_TOKEN` 为你的个人 Token 即可运行。Token 可在 [Tushare Pro 个人中心](https://tushare.pro/user/token) 获取。